In [4]:
import polars as pl
# path = "/home/jovyan/work/data/"
path = "/home/diego/BigData/financial-fraud-MLOps/data/"

In [9]:
!ls $path/raw

cards_data.csv	 mcc_codes.json		    transactions_data.csv
cities15000.txt  train_fraud_labels.json    users_data.csv
countryInfo.txt  train_fraud_labels.ndjson  ZIP_US.txt


In [10]:
transactions_data_schema = {
    "id": pl.String,
    "date": pl.String,

    "client_id": pl.String,
    "card_id": pl.String,
    "amount": pl.String,

    "use_chip": pl.String,
    "merchant_id": pl.Int64, 
    "merchant_city": pl.String,
    "merchant_state": pl.String,
    "zip": pl.String,
    "mcc": pl.String,
    "errors": pl.String
}

transactions_categorical = ["use_chip", "errors"]

transactions_data_df = (
    pl.scan_csv(f"{path}/raw/transactions_data.csv", schema=transactions_data_schema)
    .with_columns([
        pl.col(transactions_categorical).cast(pl.Categorical),
        pl.col("date").str.to_datetime("%Y-%m-%d %H:%M:%S"),
        pl.col("amount").str.replace(r"\$", "", literal=False).cast(pl.Float64),
        pl.col("zip").str.replace(r"\.0$", "", literal=False).str.pad_start(5, "0")
    ])
)

transactions_data_df.limit(5).collect()


id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
str,datetime[μs],str,str,f64,cat,i64,str,str,str,str,cat
"""7475327""",2010-01-01 00:01:00,"""1556""","""2972""",-77.0,"""Swipe Transaction""",59935,"""Beulah""","""ND""","""58523""","""5499""",null
"""7475328""",2010-01-01 00:02:00,"""561""","""4575""",14.57,"""Swipe Transaction""",67570,"""Bettendorf""","""IA""","""52722""","""5311""",null
"""7475329""",2010-01-01 00:02:00,"""1129""","""102""",80.0,"""Swipe Transaction""",27092,"""Vista""","""CA""","""92084""","""4829""",null
"""7475331""",2010-01-01 00:05:00,"""430""","""2860""",200.0,"""Swipe Transaction""",27092,"""Crown Point""","""IN""","""46307""","""4829""",null
"""7475332""",2010-01-01 00:06:00,"""848""","""3915""",46.41,"""Swipe Transaction""",13051,"""Harwood""","""MD""","""20776""","""5813""",null


In [3]:
users_data_schema = {
    "client_id": pl.String,
    "current_age": pl.Int64,
    "retirement_age": pl.Int64,
    "birth_year": pl.Int64,
    "birth_month": pl.Int64,

    "gender": pl.String,
    "address": pl.String,

    "client_latitude": pl.Float64,
    "client_longitude": pl.Float64, 

    "per_capita_income": pl.String,
    "yearly_income": pl.String,
    "total_debt": pl.String,

    "credit_score": pl.Int64,
    "num_credit_cards": pl.Int64  
}

dollar_columns = ["per_capita_income", "yearly_income", "total_debt"]

users_data_df = (
    pl.scan_csv(f"{path}/raw/users_data.csv", schema=users_data_schema)
    .with_columns([
        pl.col("gender").cast(pl.Categorical),
        pl.col(dollar_columns).str.replace(r"\$", "", literal=False).cast(pl.Float64)
    ])
)

users_data_df.limit(5).collect()

client_id,current_age,retirement_age,birth_year,birth_month,gender,address,client_latitude,client_longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
str,i64,i64,i64,i64,cat,str,f64,f64,f64,f64,f64,i64,i64
"""825""",53,66,1966,11,"""Female""","""462 Rose Lane""",34.15,-117.76,29278.0,59696.0,127613.0,787,5
"""1746""",53,68,1966,12,"""Female""","""3606 Federal Boulevard""",40.76,-73.74,37891.0,77254.0,191349.0,701,5
"""1718""",81,67,1938,11,"""Female""","""766 Third Drive""",34.02,-117.89,22681.0,33483.0,196.0,698,5
"""708""",63,63,1957,1,"""Female""","""3 Madison Street""",40.71,-73.99,163145.0,249925.0,202328.0,722,4
"""1164""",43,70,1976,9,"""Male""","""9620 Valley Stream Drive""",37.76,-122.44,53797.0,109687.0,183855.0,675,1


In [4]:
cards_data_schema = {
    "card_id": pl.String,
    "client_id": pl.String,
    "card_brand": pl.String,
    "card_type": pl.String,
    "card_number": pl.String,

    "expires": pl.String,
    "cvv": pl.String,
    "has_chip": pl.String,
    "num_cards_issued": pl.Int64,

    "credit_limit": pl.String,

    "acct_open_date": pl.String, 
    "year_pin_last_changed": pl.Int64,
    "card_on_dark_web": pl.String           
}

cards_categorical = ["card_brand", "card_type"]
cards_booleans = ["has_chip", "card_on_dark_web"]

cards_data_df = (
    pl.scan_csv(f"{path}/raw/cards_data.csv", schema=cards_data_schema)
    .with_columns([
        pl.col(cards_categorical).cast(pl.Categorical),
        pl.col(cards_booleans).str.strip_chars().str.to_uppercase() == "YES",
        pl.col("credit_limit").str.replace(r"\$", "", literal=False).cast(pl.Float64),
        (pl.lit("01/") + pl.col("expires")).str.to_date("%d/%m/%Y").alias("expires"),
        (pl.lit("01/") + pl.col("acct_open_date")).str.to_date("%d/%m/%Y").alias("acct_open_date")
    ])
)

cards_data_df.limit(5).collect()

card_id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
str,str,cat,cat,str,date,str,bool,i64,f64,date,i64,bool
"""4524""","""825""","""Visa""","""Debit""","""4344676511950444""",2022-12-01,"""623""",true,2,24295.0,2002-09-01,2008,false
"""2731""","""825""","""Visa""","""Debit""","""4956965974959986""",2020-12-01,"""393""",true,2,21968.0,2014-04-01,2014,false
"""3701""","""825""","""Visa""","""Debit""","""4582313478255491""",2024-02-01,"""719""",true,2,46414.0,2003-07-01,2004,false
"""42""","""825""","""Visa""","""Credit""","""4879494103069057""",2024-08-01,"""693""",false,1,12400.0,2003-01-01,2012,false
"""4659""","""825""","""Mastercard""","""Debit (Prepaid)""","""5722874738736011""",2009-03-01,"""75""",true,1,28.0,2008-09-01,2009,false


In [6]:
with open(f"{path}/raw/mcc_codes.json", "r", encoding="utf-8") as f:
    fragmento = f.read(400)
    print(fragmento)

{
    "5812": "Eating Places and Restaurants",
    "5541": "Service Stations",
    "7996": "Amusement Parks, Carnivals, Circuses",
    "5411": "Grocery Stores, Supermarkets",
    "4784": "Tolls and Bridge Fees",
    "4900": "Utilities - Electric, Gas, Water, Sanitary",
    "5942": "Book Stores",
    "5814": "Fast Food Restaurants",
    "4829": "Money Transfer",
    "5311": "Department Stores",
   


In [5]:
mcc_codes_df = (
    pl.read_json(f"{path}/raw/mcc_codes.json")
    .lazy()
    .unpivot(variable_name="mcc", value_name="mcc_description")
)

mcc_codes_df.limit(5).collect()


mcc,mcc_description
str,str
"""3395""","""Welding Repair"""
"""8931""","""Accounting, Auditing, and Book…"
"""5942""","""Book Stores"""
"""5921""","""Package Stores, Beer, Wine, Li…"
"""5719""","""Miscellaneous Home Furnishing …"


In [8]:
with open(f"{path}/raw/train_fraud_labels.json", "r", encoding="utf-8") as f:
    fragmento = f.read(400)
    print(fragmento)

{"target": {"10649266": "No", "23410063": "No", "9316588": "No", "12478022": "No", "9558530": "No", "12532830": "No", "19526714": "No", "9906964": "No", "13224888": "No", "13749094": "No", "12303776": "No", "19480376": "No", "11716050": "No", "20025400": "No", "7661688": "No", "16662807": "No", "21419778": "No", "18011186": "No", "23289598": "No", "11644547": "No", "23235120": "No", "19748218": "N


In [ ]:
##Ejecutamos el siguiente codigo en la terminal para transformar el json a ndjson
#jq -c '.target | to_entries[] | {id: .key, target: .value}' train_fraud_labels.json > train_fraud_labels.ndjson

In [9]:
with open(f"{path}/raw/train_fraud_labels.ndjson", "r", encoding="utf-8") as f:
    fragmento = f.read(400)
    print(fragmento)

{"id":"10649266","target":"No"}
{"id":"23410063","target":"No"}
{"id":"9316588","target":"No"}
{"id":"12478022","target":"No"}
{"id":"9558530","target":"No"}
{"id":"12532830","target":"No"}
{"id":"19526714","target":"No"}
{"id":"9906964","target":"No"}
{"id":"13224888","target":"No"}
{"id":"13749094","target":"No"}
{"id":"12303776","target":"No"}
{"id":"19480376","target":"No"}
{"id":"11716050","t


In [6]:
train_fraud_labels_df = (
    pl.scan_ndjson(f"{path}/raw/train_fraud_labels.ndjson")
    .with_columns(
        (pl.col("target").str.strip_chars().str.to_uppercase() == "YES")
    )
)

train_fraud_labels_df.limit(5).collect()

id,target
str,bool
"""10649266""",false
"""23410063""",false
"""9316588""",false
"""12478022""",false
"""9558530""",false


Las coordenadas de los códigos postales de USa los conseguimos de:  
https://download.geonames.org/export/zip/  
Debido a que hay duplicados, usaremos accuracy para seleccionar en zips y population para seleccionar en cities

In [7]:
ZIP_columns = [
    "country_code", "zip", "place_name", "admin_name1", 
    "admin_code1", "admin_name2", "admin_code2", "admin_name3",
    "admin_code3", "merchant_latitude", "merchant_longitude", "accuracy"
]

ZIP_schema = {
    "zip": pl.String,
    "merchant_latitude": pl.Float64,
    "merchant_longitude": pl.Float64
}

ZIP_US_df = (
    pl.scan_csv(f"{path}/raw/ZIP_US.txt", separator="\t", has_header = False,
                new_columns = ZIP_columns, schema_overrides= ZIP_schema,
    )
    .select(["zip", "merchant_latitude", "merchant_longitude", "accuracy"])
    .sort("accuracy", descending=True)
    .group_by("zip")
    .agg([
        pl.col("merchant_latitude").first(),
        pl.col("merchant_longitude").first()
    ])
)

ZIP_US_df.show(10)

zip,merchant_latitude,merchant_longitude
str,f64,f64
"""35957""",34.1921,-86.1941
"""35205""",33.4951,-86.8059
"""34032""",-16.5,-68.15
"""35188""",33.2068,-87.15
"""36260""",33.6031,-85.9608
"""36062""",31.8488,-86.2077
"""09070""",49.4424,10.9541
"""09627""",37.4017,14.9224
"""72456""",36.4317,-90.2751


Las coordenadas de las ciudades (necesario para las ciuades extranjeras, las cuales no tienen código postal) y el código por país las conseguimos de:  
https://download.geonames.org/export/dump/


In [8]:
cities_columns = [
    "geonameid", "name", "merchant_city", "alternatenames", 
    "city_latitude", "city_longitude", "feature class", "feature code", 
    "country_code", "cc2", "admin1 code", "admin2 code", 
    "admin3 code", "admin4 code", "population", "elevation", 
    "dem", "timezone", "modification date"
]

cities_schema = {
    "merchant_city": pl.String,
    "city_latitude": pl.Float64,
    "city_longitude": pl.Float64,
    "country_code": pl.String
}

cities_df = (
    pl.scan_csv(f"{path}/raw/cities15000.txt", separator="\t", has_header = False,
                new_columns = cities_columns, schema_overrides= ZIP_schema,
    )
    .select(["merchant_city", "city_latitude", "city_longitude", "country_code", "population"])
)

countries_df = (
    pl.scan_csv(f"{path}/raw/countryInfo.txt", separator="\t")
    .select("ISO", "Country")
    .rename({"ISO": "country_code", "Country": "merchant_state"})
)

city_country_df = (
    countries_df
    .join(cities_df, on="country_code", how="left")
    .sort("population", descending=True)
    .group_by(["merchant_state", "merchant_city"])
    .agg(
        pl.col("city_latitude").first(),
        pl.col("city_longitude").first(),
    )
)

city_country_df.show(5)

merchant_state,merchant_city,city_latitude,city_longitude
str,str,f64,f64
"""South Korea""","""Changwon""",35.22806,128.68111
"""Taiwan""","""Taoyuan""",24.99368,121.29696
"""China""","""Jinhua""",29.10678,119.64421
"""Vietnam""","""Ho Chi Minh City""",10.82302,106.62965
"""China""","""Yichang""",30.71444,111.28472


In [9]:
complete_fraud_data = (
    transactions_data_df
        .join(ZIP_US_df, on="zip", how="left")
        .join(city_country_df, on=["merchant_state", "merchant_city"], how="left")
        .with_columns(
            merchant_latitude = pl.coalesce(["merchant_latitude", "city_latitude"]),
            merchant_longitude = pl.coalesce(["merchant_longitude", "city_longitude"])
        )
        .drop("city_latitude", "city_longitude")
        .join(mcc_codes_df, on="mcc", how="left")
        .join(users_data_df, on="client_id", how="left")
        .join(cards_data_df, on=["client_id", "card_id"], how="left")
        .join(train_fraud_labels_df, on="id", how="left")        
)

complete_fraud_data.show(5)

id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors,merchant_latitude,merchant_longitude,mcc_description,current_age,retirement_age,birth_year,birth_month,gender,address,client_latitude,client_longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web,target
str,datetime[μs],str,str,f64,cat,i64,str,str,str,str,cat,f64,f64,str,i64,i64,i64,i64,cat,str,f64,f64,f64,f64,f64,i64,i64,cat,cat,str,date,str,bool,i64,f64,date,i64,bool,bool
"""7475329""",2010-01-01 00:02:00,"""1129""","""102""",80.0,"""Swipe Transaction""",27092,"""Vista""","""CA""","""92084""","""4829""",null,33.2131,-117.2243,"""Money Transfer""",49,65,1970,4,"""Male""","""2379 Forest Lane""",33.18,-117.29,16894.0,34449.0,36540.0,686,3,"""Mastercard""","""Debit""","""5874992802287595""",2020-05-01,"""256""",true,1,14802.0,2006-01-01,2008,false,false
"""7475332""",2010-01-01 00:06:00,"""848""","""3915""",46.41,"""Swipe Transaction""",13051,"""Harwood""","""MD""","""20776""","""5813""",null,38.8582,-76.6145,"""Drinking Places (Alcoholic Bev…",51,69,1968,5,"""Male""","""166 River Drive""",38.86,-76.6,33529.0,68362.0,96182.0,711,2,"""Visa""","""Debit""","""4354185735186651""",2020-01-01,"""120""",true,1,19113.0,2009-07-01,2014,false,false
"""7475328""",2010-01-01 00:02:00,"""561""","""4575""",14.57,"""Swipe Transaction""",67570,"""Bettendorf""","""IA""","""52722""","""5311""",null,41.5509,-90.4942,"""Department Stores""",48,67,1971,6,"""Male""","""604 Pine Street""",40.8,-91.12,18076.0,36853.0,112139.0,834,5,"""Mastercard""","""Credit""","""5175842699412235""",2024-12-01,"""438""",true,1,9100.0,2005-09-01,2015,false,false
"""7475327""",2010-01-01 00:01:00,"""1556""","""2972""",-77.0,"""Swipe Transaction""",59935,"""Beulah""","""ND""","""58523""","""5499""",null,47.2707,-101.8075,"""Miscellaneous Food Stores""",30,67,1989,7,"""Female""","""594 Mountain View Street""",46.8,-100.76,23679.0,48277.0,110153.0,740,4,"""Mastercard""","""Debit (Prepaid)""","""5497590243197280""",2022-07-01,"""306""",true,2,55.0,2008-05-01,2008,false,false
"""7475331""",2010-01-01 00:05:00,"""430""","""2860""",200.0,"""Swipe Transaction""",27092,"""Crown Point""","""IN""","""46307""","""4829""",null,41.4236,-87.3556,"""Money Transfer""",52,67,1967,5,"""Female""","""903 Hill Boulevard""",41.42,-87.35,26168.0,53350.0,128676.0,685,5,"""Mastercard""","""Debit""","""5346827663529174""",2024-10-01,"""54""",false,2,37634.0,2004-05-01,2006,false,null


Verificamos que tras los joins tengamos el mismo número de filas que de ids

In [10]:
complete_fraud_data_df.select(
    pl.len().alias("rows"),
    pl.col("id").n_unique().alias("unique_ids")
).collect()

rows,unique_ids
u32,u32
13305915,13305915


In [11]:
complete_fraud_data.sink_parquet(f"{path}/bronze/Polars/complete_fraud_data.parquet")